# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulRaheem2004/ML_Week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data Contract Plain-English Summary (5 Answers):

1. **Unit of Analysis (Grain):** One row represents daily search & analytics performance for one pseudonymized content item (`content_hash_id`) belonging to a specific client (`client_hash_id`) on a single date (`report_date`).
2. **Tables Used:** `fact_content_daily_performance` (partitioned daily performance data), `dim_clients` (client metadata & history start dates), and `dim_content` (content item metadata).
3. **Time Window:** Mid-panel month of **March 2026** (`2026-03-01` to `2026-03-31`), chosen to iterate safely without touching the final test outcome window (`June 2026`).
4. **Prediction Target / Proxy:** Predicting organic traffic decline or high engagement proxy (whether future organic clicks meet performance threshold relative to past window).
5. **Deliberately Excluded:** `fact_content_daily_performance_sample` (the final month June 2026) during iteration to prevent evaluating inside the test window; direct client string identifiers (used for grouping/joining only).

In [ ]:
# Code setup: DuckDB connection to Hugging Face release for mid-panel month March 2026
import os
import getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

print("DuckDB connected successfully.")
print("Selected mid-panel partition: month=2026-03 (March 2026)")


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification Bucket Table:

| Bucket | Field Name | Description & Safety Justification |
| :--- | :--- | :--- |
| **Feature** | `gsc_impressions` | Search impressions in historical window; knowable at decision moment. |
| **Feature** | `gsc_clicks` | Search clicks in historical window; knowable at decision moment. |
| **Feature** | `gsc_avg_position` | Average SERP rank position; knowable at decision moment. |
| **Feature** | `ga4_sessions` | Total sessions logged in GA4; knowable at decision moment. |
| **Feature** | `ga4_engaged_sessions` | Engaged user sessions from GA4; knowable at decision moment. |
| **Label / Proxy** | `target_label` | High performance proxy (e.g. cumulative clicks > threshold in outcome window). |
| **Context** | `client_hash_id` | Client pseudonym for grouped splits & joins only (never input feature). |
| **Context** | `content_hash_id` | Content item identifier for joining & aggregation only. |
| **Context** | `report_date` | Log date used for windowing calculations. |
| **Excluded** | `fact_daily_sample` | Final month (June 2026) data; excluded to avoid outcome window leakage during dev. |
| **Excluded** | `future_clicks` | Future click counts; excluded from features to prevent target leakage. |
| **Excluded** | `ga4_data_available` | Boolean availability flag; used strictly for filtering rows (`IS TRUE`). |

In [ ]:
# Code check: Verify column names and data types in March 2026 partition
print("=== Column Schema Inspection ===")
df_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()
print(df_schema[['column_name', 'column_type']].to_string(index=False))


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification & Feature Frame Plan:
1. **Query 1 (Grain Verification):** Assert `HAVING COUNT(*) > 1` on `(client_hash_id, content_hash_id, report_date)` returns 0 rows.
2. **Query 2 (Counts & Dates):** Report total row count, `MIN(report_date)`, and `MAX(report_date)` for March 2026.
3. **Query 3 (Availability Check):** Check row count surviving `ga4_data_available IS TRUE` filter.
4. **5-Feature Frame:**
   - `gsc_impressions_30d`: *Knowable at decision moment because accumulated during past 30 days prior to prediction cutoff.*
   - `gsc_clicks_30d`: *Knowable at decision moment because accumulated during past 30 days prior to prediction cutoff.*
   - `gsc_ctr_30d`: *Knowable at decision moment because calculated strictly from historical clicks and impressions.*
   - `gsc_avg_pos_30d`: *Knowable at decision moment because averaged from daily SERP position logs in history window.*
   - `ga4_sessions_30d`: *Knowable at decision moment because recorded in analytics during past 30 days.*
5. **The Leakage Trap Experiment:**
   - Deliberately include 1 label-derived feature (`leaked_future_clicks`), train a model, and observe artificially inflated accuracy (~1.0).
   - Remove the leaked feature, retrain on honest features, and record the true baseline score.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("--- Query 1: Verify Grain (Zero duplicates expected) ---")
q1_grain = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate rows found: {len(q1_grain)} (Grain holds true!)")

print("\n--- Query 2: Counts & Date Span (March 2026) ---")
q2_counts = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
""").df()
print(q2_counts.to_string(index=False))

print("\n--- Query 3: Availability Probe (Filtering with IS TRUE) ---")
q3_avail = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_available_rows,
        ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) AS ga4_available_pct
    FROM {TABLES['fact_daily']}
""").df()
print(q3_avail.to_string(index=False))

print("\n--- Building 5-Feature Frame & Running Leakage Trap Experiment ---")
feature_df = con.sql(f"""
    WITH march_summary AS (
        SELECT 
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS gsc_impressions_30d,
            SUM(gsc_clicks) AS gsc_clicks_30d,
            CASE WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) / SUM(gsc_impressions) ELSE 0 END AS gsc_ctr_30d,
            AVG(gsc_avg_position) AS gsc_avg_pos_30d,
            SUM(ga4_sessions) AS ga4_sessions_30d,
            
            -- Target proxy
            CASE WHEN SUM(gsc_clicks) > 10 THEN 1 ELSE 0 END AS target_label,
            
            -- TRAP: Leaked feature derived directly from label
            SUM(gsc_clicks) * 1.05 + 2 AS leaked_future_clicks
        FROM {TABLES['fact_daily']}
        WHERE ga4_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) > 0
        LIMIT 5000
    )
    SELECT * FROM march_summary
""").df()

feature_cols = ['gsc_impressions_30d', 'gsc_clicks_30d', 'gsc_ctr_30d', 'gsc_avg_pos_30d', 'ga4_sessions_30d']
X_honest = feature_df[feature_cols].fillna(0)
X_leaked = feature_df[feature_cols + ['leaked_future_clicks']].fillna(0)
y = feature_df['target_label']

# Fit model WITH leaked feature
clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaked.fit(X_leaked, y)
score_leaked = accuracy_score(y, clf_leaked.predict(X_leaked))

# Fit model WITHOUT leaked feature (Honest)
clf_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_honest.fit(X_honest, y)
score_honest = accuracy_score(y, clf_honest.predict(X_honest))

print(f"Score WITH Leaked Feature (Artificially High Trap): {score_leaked:.4f}")
print(f"Score WITHOUT Leaked Feature (Honest Baseline Score): {score_honest:.4f}")
print("Leaked feature dropped cleanly!")


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Slice Limitation:
1. **Unbalanced History & GA4 Availability Gaps:** History depth varies across clients (`gsc_data_start` vs `ga4_data_start`). Early client history contains zero-filled GA4 fields where `ga4_data_available = FALSE`. Filtering with `ga4_data_available IS TRUE` is mandatory to avoid mistaking missing instrumentation history for zero user engagement.
2. **Zero-Position Artifacts:** In Google Search Console data, `gsc_avg_position = 0` signifies missing search rank data rather than position zero. Aggregations must handle or filter 0 values explicitly.

In [ ]:
# Code check: Demonstrate client history variations in dim_clients
limitation_query = con.sql(f"""
    SELECT 
        COUNT(client_hash_id) AS total_clients,
        MIN(gsc_data_start) AS min_gsc_start,
        MAX(gsc_data_start) AS max_gsc_start,
        COUNT(CASE WHEN ga4_data_start IS NULL THEN 1 END) AS clients_missing_ga4_start
    FROM {TABLES['dim_clients']}
""").df()

print("=== Client History Depth & GA4 Availability Limits ===")
print(limitation_query.to_string(index=False))


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.